# Workshop - Redis Search: do Cache à Busca Vetorial

![Redis](https://cdn.sanity.io/images/sy1jschh/production/b4cdf336c42284d92034fffb2b14d8a79518ae82-621x456.svg)

Bem-vind[ao]s! Neste notebook vamos evoluir, passo a passo, do uso mais simples do Redis (um cache de chave/valor) até uma busca semântica completa com vetores, passando por documentos JSON, criação de índice e diferentes formas de consulta com `FT.SEARCH`.

Para uma experiência completa, recomendamos abrir o **Redis Insight** (App ou Web) em paralelo, para visualizar os dados sendo criados e consultados em tempo real.

https://redis.io/insight/

## Objetivos deste notebook

- Diferenciar um cache simples (String) de um documento estruturado (JSON)
- Armazenar documentos JSON no Redis
- Criar um índice de busca com `FT.CREATE` sobre esses documentos
- Executar `FT.SEARCH` com 3 combinações diferentes de filtros (textual, numérico, tags/negação)
- Gerar embeddings localmente, armazená-los e fazer uma busca vetorial semântica

## Pré-requisitos

Antes de rodar este notebook, cadastre os secrets abaixo no Colab (ícone de 🔑 **Secrets** no menu lateral esquerdo), com acesso liberado para este notebook:

- `REDIS_HOST`
- `REDIS_PORT`
- `REDIS_PASSWORD`

Use os dados de conexão do seu banco Redis Cloud (ou outra instância Redis com o módulo Search habilitado).

## Setup do Ambiente

In [ ]:
# Redis client + lib para gerar embeddings localmente (sem custo de API externa)
!pip install -q redis sentence-transformers

# Baixamos o modelo de embeddings já aqui no setup, pra não pagar esse custo de download
# no meio do workshop. all-MiniLM-L6-v2 é pequeno (~80MB) e roda bem em CPU.
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
EMBEDDING_DIM = embedding_model.get_sentence_embedding_dimension()
print(f"Modelo carregado. Dimensão do embedding: {EMBEDDING_DIM}")

## Setup da Conexão

In [ ]:
from google.colab import userdata

REDIS_HOST = userdata.get("REDIS_HOST")
REDIS_PORT = int(userdata.get("REDIS_PORT"))
REDIS_PASSWORD = userdata.get("REDIS_PASSWORD")

## Teste de Conexão

In [ ]:
import redis

r = redis.Redis(
    host=REDIS_HOST,
    port=REDIS_PORT,
    password=REDIS_PASSWORD,
    decode_responses=True,
)

print("Conexão ok:", r.ping())

# Parte Um - Cache Básico

No fim das contas, um cache é só um par chave/valor, com ou sem expiração (TTL). Vamos começar por aqui antes de subir a régua.

In [ ]:
# Um contador simples, sem TTL
r.delete("catalogo:visitas")
r.incr("catalogo:visitas")
r.incr("catalogo:visitas")
print("Visitas ao catálogo:", r.get("catalogo:visitas"))

# Um cache com expiração (TTL) - útil pra guardar o resultado de uma consulta cara por alguns segundos
r.set("catalogo:destaque", "Fone Bluetooth XPTO", ex=30)
print("Destaque:", r.get("catalogo:destaque"))
print("TTL restante (segundos):", r.ttl("catalogo:destaque"))

# Parte Dois - Armazenando um Documento JSON

Uma String é ótima para um valor simples, mas não representa bem uma entidade com vários campos (nome, preço, categoria...). Para isso usamos `JSON.SET`, guardando cada produto como um documento JSON sob o prefixo `product:`.

In [ ]:
produtos = [
    {"sku": "SKU1001", "name": "Fone de Ouvido Bluetooth XPTO", "category": "eletronicos", "brand": "xpto",
     "price": 149.90, "stock": 35,
     "description": "Fone de ouvido sem fio com cancelamento de ruído, ideal para treinos e viagens longas."},
    {"sku": "SKU1002", "name": "Caixa de Som Portátil SoundGo", "category": "eletronicos", "brand": "soundgo",
     "price": 199.90, "stock": 12,
     "description": "Caixa de som à prova d'água, perfeita para levar à praia ou piscina."},
    {"sku": "SKU1003", "name": "Smartwatch FitPulse", "category": "eletronicos", "brand": "fitpulse",
     "price": 349.00, "stock": 8,
     "description": "Relógio inteligente com monitor de batimentos cardíacos e GPS integrado para corrida."},
    {"sku": "SKU1004", "name": "Tênis de Corrida AeroRun", "category": "esporte", "brand": "aerorun",
     "price": 279.90, "stock": 40,
     "description": "Tênis leve e amortecido, desenvolvido para corridas de longa distância na rua."},
    {"sku": "SKU1005", "name": "Bicicleta Urbana CityBike", "category": "esporte", "brand": "citybike",
     "price": 899.00, "stock": 5,
     "description": "Bicicleta confortável para deslocamentos urbanos e passeios tranquilos no fim de semana."},
    {"sku": "SKU1006", "name": "Liquidificador PowerMix", "category": "casa", "brand": "powermix",
     "price": 129.90, "stock": 22,
     "description": "Liquidificador potente, ideal para preparar vitaminas e sucos no dia a dia."},
    {"sku": "SKU1007", "name": "Aspirador Robô CleanBot", "category": "casa", "brand": "cleanbot",
     "price": 899.90, "stock": 3,
     "description": "Aspirador robô autônomo que mapeia a casa e limpa sozinho enquanto você trabalha."},
    {"sku": "SKU1008", "name": "Cafeteira Elétrica BrewTime", "category": "casa", "brand": "brewtime",
     "price": 219.00, "stock": 18,
     "description": "Cafeteira programável para preparar café fresquinho assim que você acorda."},
    {"sku": "SKU1009", "name": "Headset Gamer SoundGo Pro", "category": "eletronicos", "brand": "soundgo",
     "price": 259.90, "stock": 15,
     "description": "Headset com som surround, indicado para longas sessões de jogos online."},
    {"sku": "SKU1010", "name": "Colchonete de Yoga FlexMat", "category": "esporte", "brand": "flexmat",
     "price": 89.90, "stock": 50,
     "description": "Colchonete antiderrapante para práticas de yoga e alongamento em casa."},
]

for produto in produtos:
    chave = f"product:{produto['sku']}"
    r.json().set(chave, "$", produto)

print(f"{len(produtos)} produtos armazenados como documentos JSON")
print(r.json().get("product:SKU1001"))

# Parte Três - Criando o Índice

Com os documentos no lugar, criamos um índice com `FT.CREATE` apontando para o prefixo `product:`. A escolha do tipo de campo importa:

- `TEXT` para campos de busca textual livre (`name`, `description`)
- `TAG` para valores de match exato usados em filtros (`category`, `brand`)
- `NUMERIC` para faixas e ordenação (`price`, `stock`)

In [ ]:
from redis.commands.search.field import TextField, TagField, NumericField
from redis.commands.search.index_definition import IndexDefinition, IndexType

INDEX_NAME = "idx:products"

# Torna a célula idempotente - se o índice já existir, recriamos do zero
try:
    r.ft(INDEX_NAME).dropindex()
except Exception:
    pass  # índice ainda não existe, sem problema

schema = (
    TextField("$.name", as_name="name", weight=2.0),
    TextField("$.description", as_name="description"),
    TagField("$.category", as_name="category"),
    TagField("$.brand", as_name="brand"),
    NumericField("$.price", as_name="price", sortable=True),
    NumericField("$.stock", as_name="stock", sortable=True),
)

r.ft(INDEX_NAME).create_index(
    schema,
    definition=IndexDefinition(prefix=["product:"], index_type=IndexType.JSON),
)

info = r.ft(INDEX_NAME).info()
print("Índice criado. Documentos indexados:", info["num_docs"])

# Parte Quatro - Buscando com FT.SEARCH

Com o índice pronto, vamos ver 3 formas distintas de consultar os mesmos dados. Sempre com `DIALECT 2`, que é o obrigatório para a sintaxe mais moderna (e para vetores, mais adiante).

### Exemplo 1 — Busca textual combinada com filtro de categoria (TEXT + TAG)

In [ ]:
from redis.commands.search.query import Query

query = (
    Query("bluetooth @category:{eletronicos}")
    .return_fields("name", "category", "price")
    .dialect(2)
)
resultado = r.ft(INDEX_NAME).search(query)

print("Total encontrado:", resultado.total)
for doc in resultado.docs:
    print(f"- {doc.name} [{doc.category}] - R$ {doc.price}")

### Exemplo 2 — Faixa de preço, ordenada e paginada (NUMERIC + SORTBY + LIMIT)

In [ ]:
query = (
    Query("@price:[100 300]")
    .sort_by("price", asc=True)
    .paging(0, 5)
    .return_fields("name", "price", "category")
    .dialect(2)
)
resultado = r.ft(INDEX_NAME).search(query)

print("Produtos entre R$100 e R$300 (mais baratos primeiro):")
for doc in resultado.docs:
    print(f"- {doc.name} (R$ {doc.price}) [{doc.category}]")

### Exemplo 3 — Combinando OR, negação e filtro numérico (TAG OR + NOT + NUMERIC)

In [ ]:
# Produtos das categorias "casa" OU "esporte", que NÃO sejam da marca CityBike,
# e que estejam com estoque baixo (<=10) - útil pra identificar itens quase esgotados
query = (
    Query("(@category:{casa|esporte}) -@brand:{citybike} @stock:[0 10]")
    .return_fields("name", "brand", "category", "stock")
    .dialect(2)
)
resultado = r.ft(INDEX_NAME).search(query)

print("Produtos quase esgotados (casa/esporte, exceto CityBike):")
for doc in resultado.docs:
    print(f"- {doc.name} ({doc.brand}) - estoque: {doc.stock}")

# Parte Cinco - Busca Vetorial (Semântica)

Filtros exatos e texto tokenizado resolvem muita coisa, mas não capturam *significado*. Para isso, geramos um vetor (embedding) para cada descrição e buscamos por similaridade — mesmo quando as palavras usadas na busca não aparecem literalmente no texto.

### Passo 1 — Gerar e armazenar o vetor de cada produto

In [ ]:
import numpy as np

for produto in produtos:
    chave = f"product:{produto['sku']}"
    vetor = embedding_model.encode(produto["description"]).astype(np.float32)
    # Campos vetoriais em documentos JSON são armazenados como uma lista de floats
    r.json().set(chave, "$.embedding", vetor.tolist())

print("Embeddings gerados e armazenados em cada documento")

### Passo 2 — Adicionar o campo vetorial ao índice existente

Não precisamos recriar o índice do zero: `FT.ALTER` adiciona o novo campo aos documentos já existentes, em background.

In [ ]:
import time
from redis.commands.search.field import VectorField

r.ft(INDEX_NAME).alter_schema_add(
    VectorField(
        "$.embedding",
        "HNSW",
        {"TYPE": "FLOAT32", "DIM": EMBEDDING_DIM, "DISTANCE_METRIC": "COSINE"},
        as_name="embedding",
    )
)

# Dá um respiro pro índice terminar de escanear os documentos existentes.
# Em produção, prefira checar info()["percent_indexed"] em loop em vez de um sleep fixo.
time.sleep(1)
print("Campo vetorial adicionado ao índice")

### Passo 3 — Comparando busca textual x busca semântica

Vamos procurar por algo que **não aparece literalmente** em nenhuma descrição, para deixar clara a diferença.

In [ ]:
# Busca textual pela palavra "varrer" - não existe em nenhuma descrição
query_textual = Query("varrer").dialect(2)
resultado_textual = r.ft(INDEX_NAME).search(query_textual)
print("Busca textual por 'varrer':", resultado_textual.total, "resultado(s)")

In [ ]:
consulta = "não quero mais perder tempo varrendo a casa"
#consulta = "não sei fazer café"
#consulta = "gosto de ouvir musica quando estou na praia"

vetor_consulta = embedding_model.encode(consulta).astype(np.float32).tobytes()

query_vetorial = (
    Query("(*)=>[KNN 1 @embedding $vetor_consulta AS score]")
    .sort_by("score")
    .return_fields("name", "description", "score")
    .dialect(2)
)
resultado_vetorial = r.ft(INDEX_NAME).search(
    query_vetorial, query_params={"vetor_consulta": vetor_consulta}
)

print(f"Busca semântica por: '{consulta}'\n")
for doc in resultado_vetorial.docs:
    print(f"- {doc.name} (score: {float(doc.score):.4f})")
    print(f"  {doc.description}\n")

# Recapitulando

Neste notebook, evoluímos em camadas:

- **Cache:** String simples, com e sem TTL
- **Documentos:** `JSON.SET` para guardar entidades com múltiplos campos
- **Índice:** `FT.CREATE` sobre JSON, escolhendo `TEXT`/`TAG`/`NUMERIC` para cada campo
- **FT.SEARCH:** filtros textuais, numéricos, negação e combinações de tags
- **Vetores:** geração de embeddings local, `FT.ALTER` para adicionar o campo vetorial, e busca por `KNN`

Este notebook faz parte de uma série de treinamento em Redis. Os outros módulos são:

- `search.ipynb` — Cache, JSON, índices e busca (textual e vetorial) com Redis Search
- `streams.ipynb` — Filas, Pub/Sub e Streams para mensageria
- `agent_memory.ipynb` — Memória de curto e longo prazo para agentes de IA
- `langcache.ipynb` — Cache semântico de respostas de LLM com LangCache